<a href="https://colab.research.google.com/github/shyamgadhiya/ML-DL-Tasks/blob/main/Sentiment_Analysis_of_IMDB_Using_LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Sentiment Analysis on IMDB Movies Reveiws dataset**

## The dataset is already converted in to numbers using Word Indexing (Every word as integer) so we can use in our model.


In [1]:
import numpy as np
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense,SpatialDropout1D, Dropout

# 1. Load the data (keep only the 10,000 most common words)
print("Loading data...")
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=10000)

# 2. Pad the sequences to 200 words
max_length = 200
X_train = pad_sequences(X_train, maxlen=max_length)
X_test = pad_sequences(X_test, maxlen=max_length)

print(f"Data shaped: {X_train.shape}")

Loading data...
Data shaped: (25000, 200)


## We have implemented padding for each sequence because every reviews has different lengths so each review padded to max 200 words.


In [2]:
model = Sequential()

# Input dimension is 10,000 words, outputting vectors of size 128
model.add(Embedding(input_dim=10000, output_dim=128, input_length=max_length))

# The LSTM layer with 64 units
model.add(LSTM(64))

# Output layer
model.add(Dense(1, activation='sigmoid'))

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [3]:
# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train it!
print("Starting training...")
history = model.fit(
    X_train, y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2 # Use 20% of data to check for overfitting
)

Starting training...
Epoch 1/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 9s 17ms/step - accuracy: 0.7914 - loss: 0.4387 - val_accuracy: 0.8474 - val_loss: 0.3533
Epoch 2/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.8999 - loss: 0.2569 - val_accuracy: 0.8542 - val_loss: 0.3986
Epoch 3/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.9312 - loss: 0.1875 - val_accuracy: 0.8660 - val_loss: 0.3371
Epoch 4/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 15ms/step - accuracy: 0.9553 - loss: 0.1295 - val_accuracy: 0.8736 - val_loss: 0.3850
Epoch 5/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.9634 - loss: 0.1072 - val_accuracy: 0.8216 - val_loss: 0.4644


## Now using Bidirectional LSTM for Better Result

In [9]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SpatialDropout1D, Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

model_best = Sequential([
    # 1. Removed deprecated 'input_length'
    Embedding(input_dim=10000, output_dim=128),

    # 2. Drop 30% of feature maps to prevent co-adaptation of words
    SpatialDropout1D(0.3),

    # 3. Add dropout and recurrent_dropout inside the LSTM layer
    Bidirectional(LSTM(64, dropout=0.3)),

    # 4. Dense layer with Dropout
    Dense(32, activation='relu'),
    Dropout(0.5),

    Dense(1, activation='sigmoid')
])

model_best.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

## Here we have implemented early stopping and Learning Rate scheduling for better results.

In [10]:
# Callbacks to handle training automatically
callbacks = [
    # Stops training if val_loss doesn't improve for 2 epochs, and keeps the best weights
    EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True),

    # Reduces learning rate when val_loss plateaus
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=1, min_lr=1e-5)
]

history = model_best.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=10,
    batch_size=64,
    callbacks=callbacks
)

Epoch 1/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 15s 32ms/step - accuracy: 0.7720 - loss: 0.4637 - val_accuracy: 0.8539 - val_loss: 0.3296 - learning_rate: 0.0010
Epoch 2/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 12s 31ms/step - accuracy: 0.8915 - loss: 0.2808 - val_accuracy: 0.8710 - val_loss: 0.3114 - learning_rate: 0.0010
Epoch 3/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 12s 31ms/step - accuracy: 0.9172 - loss: 0.2212 - val_accuracy: 0.8697 - val_loss: 0.3303 - learning_rate: 0.0010
Epoch 4/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 12s 31ms/step - accuracy: 0.9463 - loss: 0.1537 - val_accuracy: 0.8707 - val_loss: 0.3951 - learning_rate: 5.0000e-04


##Test Model

In [15]:
def predict_sentiment(review_text):
    # 1. Get the word-to-ID dictionary from Keras
    word_index = imdb.get_word_index()

    # 2. Convert your text to lower case and split it into words
    words = review_text.lower().split()

    # 3. Convert words to their integer IDs (defaulting to 2 for unknown words)
    tokens = [word_index.get(word, 2) + 3 for word in words]

    # 4. Pad the sequence to 200 words
    padded_tokens = pad_sequences([tokens], maxlen=200)

    # 5. Predict
    prediction = model_best.predict(padded_tokens)

    if prediction[0][0] > 0.5:
        return f"Positive ({prediction[0][0]:.2f})"
    else:
        return f"Negative ({prediction[0][0]:.2f})"

# Test it
print(predict_sentiment("The acting was brilliant and I loved the story"))
print(predict_sentiment("What a waste of time, totally boring"))


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
Positive (0.75)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
Negative (0.02)
